In [28]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 0 : CONFIGURATION                                                   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

storage_account = "energybigdatastorage"
container_raw = "raw"
container_processed = "processed"

path_raw = f"abfss://{container_raw}@{storage_account}.dfs.core.windows.net/energy_data_extracted/archive (3).zip/weather_hourly_darksky.csv"
path_processed = f"abfss://{container_processed}@{storage_account}.dfs.core.windows.net/weather_hourly/"

print(f"Source: {path_raw}")
print(f"Destination: {path_processed}")

In [29]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 1 : INGESTION                                                         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(path_raw)

print(f"Nombre de lignes: {df.count()}")
print(f"Nombre de colonnes: {len(df.columns)}")
print(f"\n=== SCHEMA ===")
df.printSchema()
print(f"\n=== APERCU ===")
df.show(3)

In [31]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 2 : PROFILING AVANT NETTOYAGE                                         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

print("=== VALEURS NULLES ===")
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()

print(f"\n=== DOUBLONS ===")
print(f"Nombre de doublons: {df.count() - df.dropDuplicates().count()}")

print(f"\n=== INSPECTION DATE ===")
df.select("time").show(5, truncate=False)

print(f"\n=== STATISTIQUES COLONNES NUMERIQUES ===")
numeric_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, (DoubleType, FloatType, IntegerType, LongType))]
print(f"Colonnes numeriques: {numeric_cols}")

for c in numeric_cols:
    stats = df.select(
        F.min(c).alias("min"),
        F.max(c).alias("max"),
        F.avg(c).alias("avg"),
        F.stddev(c).alias("stddev"),
        F.count(F.when(F.col(c).isNull(), c)).alias("nulls")
    ).collect()[0]
    print(f"{c:25s} | min={stats.min:10.2f} | max={stats.max:10.2f} | avg={stats.avg:10.2f} | std={stats.stddev:10.2f} | nulls={stats.nulls}")

print(f"\n=== COLONNES STRING (TOP 10) ===")
string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType) and f.name != "time"]
for c in string_cols:
    print(f"\n--- {c} ---")
    df.groupBy(c).count().orderBy(F.desc("count")).show(10, truncate=False)

In [32]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 3 : NETTOYAGE                                                         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# 1. Convertir timestamp
df_clean = df.withColumn("timestamp", F.to_timestamp(F.col("time"), "yyyy-MM-dd HH:mm:ss"))

# 2. Supprimer sans timestamp
null_ts = df_clean.filter(F.col("timestamp").isNull()).count()
df_clean = df_clean.filter(F.col("timestamp").isNotNull())
print(f"Lignes sans timestamp supprimees: {null_ts}")

# 3. Features temporelles
df_clean = df_clean \
    .withColumn("date", F.to_date(F.col("timestamp"))) \
    .withColumn("hour", F.hour(F.col("timestamp"))) \
    .withColumn("year", F.year(F.col("timestamp"))) \
    .withColumn("month", F.month(F.col("timestamp"))) \
    .withColumn("day", F.dayofmonth(F.col("timestamp"))) \
    .withColumn("dayofweek", F.dayofweek(F.col("timestamp"))) \
    .withColumn("is_weekend", F.when(F.col("dayofweek").isin([1, 7]), 1).otherwise(0))

# 4. Colonnes meteo a nettoyer
weather_cols = ["temperature", "apparentTemperature", "humidity", 
                "windSpeed", "windBearing", "visibility", "pressure",
                "cloudCover", "precipIntensity", "precipProbability"]
existing_cols = [c for c in weather_cols if c in df_clean.columns]
print(f"Colonnes meteo detectees: {existing_cols}")

# 5. Conversion + NaN -> NULL
for c in existing_cols:
    df_clean = df_clean.withColumn(c, F.col(c).cast("double"))
    df_clean = df_clean.withColumn(c, F.when(F.isnan(F.col(c)), F.lit(None)).otherwise(F.col(c)))

# 6. Temperatures absurdes (< -30 ou > 50) -> NULL
if "temperature" in existing_cols:
    absurd_temp = df_clean.filter((F.col("temperature") < -30) | (F.col("temperature") > 50)).count()
    df_clean = df_clean.withColumn("temperature",
        F.when((F.col("temperature") < -30) | (F.col("temperature") > 50), F.lit(None))
        .otherwise(F.col("temperature"))
    )
    print(f"Temperatures absurdes corrigees: {absurd_temp}")

# 7. Humidite [0, 1]
if "humidity" in existing_cols:
    df_clean = df_clean.withColumn("humidity",
        F.when(F.col("humidity") > 1, F.col("humidity") / 100)
        .when((F.col("humidity") < 0) | (F.col("humidity") > 1), F.lit(None))
        .otherwise(F.col("humidity"))
    )

# 8. Pression [900, 1100] hPa
if "pressure" in existing_cols:
    absurd_press = df_clean.filter((F.col("pressure") < 900) | (F.col("pressure") > 1100)).count()
    df_clean = df_clean.withColumn("pressure",
        F.when((F.col("pressure") < 900) | (F.col("pressure") > 1100), F.lit(None))
        .otherwise(F.col("pressure"))
    )
    print(f"Pressions absurdes corrigees: {absurd_press}")

# 9. Supprimer doublons timestamp
dupes = df_clean.count() - df_clean.dropDuplicates(["timestamp"]).count()
df_clean = df_clean.dropDuplicates(["timestamp"])
print(f"Doublons timestamp supprimes: {dupes}")

# 10. Trier
df_clean = df_clean.orderBy("timestamp")

# 11. Interpolation simple (moyenne fenetre glissante ±2h)
for c in existing_cols:
    window_spec = Window.orderBy("timestamp").rowsBetween(-2, 2)
    df_clean = df_clean.withColumn(f"{c}_imputed",
        F.when(F.col(c).isNull(), F.round(F.avg(F.col(c)).over(window_spec), 2))
        .otherwise(F.col(c))
    )

# 12. Metadata
df_clean = df_clean.withColumn("processed_date", F.current_date())

print(f"\nNombre de lignes nettoyees: {df_clean.count()}")
df_clean.select("timestamp", "temperature", "temperature_imputed", "humidity", "pressure").show(5)

In [33]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 4 : STATISTIQUES APRES NETTOYAGE                                      ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

print("=== VALEURS NULLES AVANT INTERPOLATION ===")
df_clean.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in existing_cols]).show()

print("\n=== VALEURS NULLES APRES INTERPOLATION ===")
df_clean.select([F.count(F.when(F.col(f"{c}_imputed").isNull(), f"{c}_imputed")).alias(f"{c}_imputed") for c in existing_cols]).show()

print("\n=== DISTRIBUTION TEMPORRELLE ===")
df_clean.select(
    F.min("timestamp").alias("ts_min"),
    F.max("timestamp").alias("ts_max"),
    F.countDistinct("date").alias("nb_jours"),
    F.countDistinct("hour").alias("nb_heures")
).show()

In [34]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 5 : SAUVEGARDE DELTA                                                  ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("year", "month") \
    .save(path_processed)

print("Sauvegarde terminee dans processed/weather_hourly/")

In [35]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 6 : VERIFICATION                                                      ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

df_verify = spark.read.format("delta").load(path_processed)
print(f"Verification: {df_verify.count()} lignes")
print(f"\n=== SCHEMA ===")
df_verify.printSchema()
print(f"\n=== APERCU ===")
df_verify.select("timestamp", "temperature", "temperature_imputed", "humidity", "pressure", "year", "month").show(5)

print(f"\n=== TESTS RAPIDES ===")
print(f"timestamp NULL: {df_verify.filter(F.col('timestamp').isNull()).count()}")
print(f"Temperatures absurdes: {df_verify.filter((F.col('temperature') < -30) | (F.col('temperature') > 50)).count()}")
print(f"Humidite hors [0,1]: {df_verify.filter((F.col('humidity') < 0) | (F.col('humidity') > 1)).count()}")
print(f"Pression hors [900,1100]: {df_verify.filter((F.col('pressure') < 900) | (F.col('pressure') > 1100)).count()}")
print(f"Doublons timestamp: {df_verify.count() - df_verify.dropDuplicates(['timestamp']).count()}")